# Aula 6 — Como avaliar um modelo de classificação?

**Trilha de IA Aplicada — Projeto Cuidadores**  
**Notebook do estudante**

Nesta aula, vamos comparar **KNN** e **Árvore de Decisão** usando mais do que a acurácia. Investigaremos:

- matriz de confusão;
- precisão;
- recall;
- F1-score;
- classes desbalanceadas.

> **Atenção:** todos os dados são sintéticos e a atividade é exclusivamente educacional. Os modelos não estão validados para decisões reais de saúde.


## Objetivos de aprendizagem

Ao final da prática, você deverá conseguir:

1. identificar VP, VN, FP e FN em uma matriz de confusão;
2. calcular e interpretar acurácia, precisão, recall e F1-score;
3. comparar modelos usando evidências diferentes;
4. reconhecer a armadilha da acurácia em classes desbalanceadas;
5. relacionar o tipo de erro ao contexto do problema.


## Roteiro da prática

Vamos reutilizar o dataset da Aula 5 para manter a continuidade:

`dados → treino/teste → KNN e Árvore → previsões → métricas → decisão`

Depois, usaremos um segundo dataset para observar o desbalanceamento de classes.


## 1. Preparando o ambiente

Execute a célula abaixo para importar as bibliotecas.


In [ ]:
import io
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

print("Ambiente preparado!")


## 2. Função para carregar os arquivos

O notebook procura o CSV no ambiente atual. Se não encontrar e estiver no Google Colab, abrirá a janela de upload.


In [ ]:
def carregar_csv(nome_arquivo):
    """Carrega um CSV local ou solicita upload quando estiver no Google Colab."""
    if os.path.exists(nome_arquivo):
        print(f"Arquivo encontrado: {nome_arquivo}")
        return pd.read_csv(nome_arquivo)

    try:
        from google.colab import files

        print(f"Selecione o arquivo: {nome_arquivo}")
        enviados = files.upload()
        if nome_arquivo not in enviados:
            nome_recebido = next(iter(enviados))
            print(f"Usando o arquivo enviado: {nome_recebido}")
            return pd.read_csv(io.BytesIO(enviados[nome_recebido]))
        return pd.read_csv(io.BytesIO(enviados[nome_arquivo]))
    except ImportError as erro:
        raise FileNotFoundError(
            f"Coloque o arquivo '{nome_arquivo}' na mesma pasta do notebook."
        ) from erro


## 3. Carregando o dataset principal


In [ ]:
dados = carregar_csv("atrasos_medicamentos.csv")

print("Dimensões do dataset:", dados.shape)
display(dados.head())


### Conferindo estrutura e qualidade

Antes de treinar um modelo, confirme tipos, valores ausentes e distribuição da variável-alvo.


In [ ]:
print("Tipos das colunas:")
display(dados.dtypes.to_frame("tipo"))

print("\nValores ausentes por coluna:")
display(dados.isna().sum().to_frame("quantidade"))

print("\nDistribuição do target:")
distribuicao_principal = dados["medicamento_atrasado"].value_counts().sort_index()
display(distribuicao_principal.rename(index={0: "0 — não atrasou", 1: "1 — atrasou"}).to_frame("registros"))


In [ ]:
ax = distribuicao_principal.plot(
    kind="bar",
    color=["#14B8A6", "#F97316"],
    figsize=(6, 4),
    rot=0,
)
ax.set_title("Distribuição das classes — dataset principal")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade de registros")
ax.set_xticklabels(["0 — não atrasou", "1 — atrasou"])
plt.tight_layout()
plt.show()


### Pare e pense

- As classes estão muito desbalanceadas?
- Um modelo que sempre previsse a classe mais frequente seria útil neste conjunto?
- Qual classe representa o caso que desejamos encontrar?


## 4. Features, target e separação treino/teste

Usaremos as mesmas duas features numéricas da Aula 5.


In [ ]:
features = ["quantidade_lembretes", "atraso_medio"]
target = "medicamento_atrasado"

X = dados[features]
y = dados[target]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Linhas de treino:", len(X_treino))
print("Linhas de teste:", len(X_teste))
print("Classes no teste:")
display(y_teste.value_counts().sort_index().to_frame("registros"))


## 5. Treinando os dois modelos

O KNN precisa que as features estejam na mesma escala. A Árvore de Decisão usa os valores originais.


In [ ]:
scaler = StandardScaler()

X_treino_escalado = scaler.fit_transform(X_treino)
X_teste_escalado = scaler.transform(X_teste)

modelo_knn = KNeighborsClassifier(n_neighbors=3)
modelo_knn.fit(X_treino_escalado, y_treino)

modelo_arvore = DecisionTreeClassifier(random_state=RANDOM_STATE)
modelo_arvore.fit(X_treino, y_treino)

previsoes_knn = modelo_knn.predict(X_teste_escalado)
previsoes_arvore = modelo_arvore.predict(X_teste)

print("KNN e Árvore de Decisão treinados!")


## 6. Primeira comparação: acurácia

A acurácia pergunta: **de todas as previsões, quantas foram corretas?**


In [ ]:
acuracia_knn = accuracy_score(y_teste, previsoes_knn)
acuracia_arvore = accuracy_score(y_teste, previsoes_arvore)

print(f"Acurácia do KNN: {acuracia_knn:.2%}")
print(f"Acurácia da Árvore: {acuracia_arvore:.2%}")


### O que a acurácia não mostrou?

Os dois modelos podem ter a mesma quantidade total de acertos e, ainda assim, cometer **tipos diferentes de erro**. Vamos abrir essa informação.


## 7. Matriz de confusão

Na classificação binária, o Scikit-Learn organiza a matriz assim:

|  | Previsto 0 | Previsto 1 |
|---|---:|---:|
| **Real 0** | VN | FP |
| **Real 1** | FN | VP |

- **VN:** previu não atraso e realmente não houve atraso;
- **FP:** previu atraso, mas não houve atraso;
- **FN:** havia atraso, mas o modelo não o encontrou;
- **VP:** previu atraso e realmente houve atraso.


In [ ]:
matriz_knn = confusion_matrix(y_teste, previsoes_knn)
matriz_arvore = confusion_matrix(y_teste, previsoes_arvore)

print("Matriz de confusão — KNN")
print(matriz_knn)

print("\nMatriz de confusão — Árvore")
print(matriz_arvore)


In [ ]:
vn_knn, fp_knn, fn_knn, vp_knn = matriz_knn.ravel()
vn_arvore, fp_arvore, fn_arvore, vp_arvore = matriz_arvore.ravel()

erros_modelos = pd.DataFrame([
    {"modelo": "KNN", "VN": vn_knn, "FP": fp_knn, "FN": fn_knn, "VP": vp_knn},
    {"modelo": "Árvore de Decisão", "VN": vn_arvore, "FP": fp_arvore, "FN": fn_arvore, "VP": vp_arvore},
])

display(erros_modelos)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ConfusionMatrixDisplay(
    confusion_matrix=matriz_knn,
    display_labels=["Não atrasou", "Atrasou"],
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("KNN")

ConfusionMatrixDisplay(
    confusion_matrix=matriz_arvore,
    display_labels=["Não atrasou", "Atrasou"],
).plot(ax=axes[1], colorbar=False, cmap="Greens")
axes[1].set_title("Árvore de Decisão")

plt.tight_layout()
plt.show()


### Interpretação no contexto

Neste experimento:

- um **falso positivo (FP)** gera um alerta de atraso quando o registro real era “não atrasou”;
- um **falso negativo (FN)** deixa de identificar um atraso que realmente existia.

**Pergunta central:** qual desses erros teria maior custo no problema estudado pelo grupo?


## 8. Precisão, recall e F1-score

- **Precisão:** quando o modelo disse “atrasou”, quantas vezes estava correto?
- **Recall:** dos atrasos que realmente existiam, quantos o modelo encontrou?
- **F1-score:** busca equilíbrio entre precisão e recall.


In [ ]:
def calcular_metricas(nome_modelo, y_real, y_previsto):
    return {
        "Modelo": nome_modelo,
        "Acurácia": accuracy_score(y_real, y_previsto),
        "Precisão": precision_score(y_real, y_previsto, zero_division=0),
        "Recall": recall_score(y_real, y_previsto, zero_division=0),
        "F1-score": f1_score(y_real, y_previsto, zero_division=0),
    }


comparacao_modelos = pd.DataFrame([
    calcular_metricas("KNN (K = 3)", y_teste, previsoes_knn),
    calcular_metricas("Árvore de Decisão", y_teste, previsoes_arvore),
]).set_index("Modelo")

comparacao_formatada = comparacao_modelos.map(lambda valor: f"{valor:.2%}")
display(comparacao_formatada)


In [ ]:
ax = comparacao_modelos.plot(
    kind="bar",
    figsize=(10, 5),
    ylim=(0, 1.08),
    color=["#0F766E", "#2563EB", "#F97316", "#7C3AED"],
    rot=0,
)
ax.set_title("Comparação das métricas")
ax.set_ylabel("Valor")
ax.legend(loc="lower right")
ax.yaxis.set_major_formatter(lambda valor, posicao: f"{valor:.0%}")
plt.tight_layout()
plt.show()


### Leia a tabela antes de escolher

Observe:

1. os modelos têm a mesma acurácia?
2. qual modelo tem maior precisão?
3. qual tem maior recall?
4. qual deixou mais falsos negativos?
5. a sua escolha mudaria conforme o custo do erro?


## 9. Classification report

O relatório apresenta métricas para cada classe. Nesta aula, concentre a interpretação especialmente na classe **1 — atrasou**.


In [ ]:
print("CLASSIFICATION REPORT — KNN\n")
print(classification_report(
    y_teste,
    previsoes_knn,
    target_names=["Não atrasou (0)", "Atrasou (1)"],
    zero_division=0,
))

print("\nCLASSIFICATION REPORT — ÁRVORE\n")
print(classification_report(
    y_teste,
    previsoes_arvore,
    target_names=["Não atrasou (0)", "Atrasou (1)"],
    zero_division=0,
))


## 10. Atividade de fixação — cálculo manual

Considere:

- VP = 30
- VN = 50
- FP = 10
- FN = 10

Execute a célula e relacione cada resultado à pergunta que a métrica responde.


In [ ]:
vp = 30
vn = 50
fp = 10
fn = 10

total = vp + vn + fp + fn
acertos = vp + vn
erros = fp + fn
acuracia_manual = acertos / total
precisao_manual = vp / (vp + fp)
recall_manual = vp / (vp + fn)
f1_manual = 2 * (precisao_manual * recall_manual) / (precisao_manual + recall_manual)

print("Total:", total)
print("Acertos:", acertos)
print("Erros:", erros)
print(f"Acurácia: {acuracia_manual:.2%}")
print(f"Precisão: {precisao_manual:.2%}")
print(f"Recall: {recall_manual:.2%}")
print(f"F1-score: {f1_manual:.2%}")


### Responda no notebook

1. Qual erro representa um “SIM” previsto incorretamente?
2. Qual erro representa um “SIM” real que não foi encontrado?
3. Neste exemplo, precisão e recall são iguais? Por quê?

**Sua resposta:**  
Escreva aqui.


## 11. Classes desbalanceadas

Agora usaremos um conjunto com:

- 90% de registros da classe 0;
- 10% de registros da classe 1.

Um “modelo” ingênuo sempre responderá 0. Vamos verificar por que os 90% de acurácia não significam bom desempenho.


In [ ]:
dados_desbalanceados = carregar_csv("atrasos_medicamentos_desbalanceado.csv")

distribuicao_desbalanceada = (
    dados_desbalanceados["medicamento_atrasado"]
    .value_counts()
    .sort_index()
)

print("Dimensões:", dados_desbalanceados.shape)
display(distribuicao_desbalanceada.rename(index={0: "0 — não atrasou", 1: "1 — atrasou"}).to_frame("registros"))

percentuais = (
    dados_desbalanceados["medicamento_atrasado"]
    .value_counts(normalize=True)
    .sort_index()
)
tabela_percentuais = percentuais.rename(
    index={0: "0 — não atrasou", 1: "1 — atrasou"}
).to_frame("percentual")
tabela_percentuais["percentual"] = tabela_percentuais["percentual"].map(lambda valor: f"{valor:.1%}")
display(tabela_percentuais)


In [ ]:
y_desbalanceado = dados_desbalanceados["medicamento_atrasado"]

# Estratégia ingênua: prever 0 para todos os registros.
previsoes_sempre_zero = np.zeros(len(y_desbalanceado), dtype=int)

metricas_ingenuas = pd.Series({
    "Acurácia": accuracy_score(y_desbalanceado, previsoes_sempre_zero),
    "Precisão": precision_score(y_desbalanceado, previsoes_sempre_zero, zero_division=0),
    "Recall": recall_score(y_desbalanceado, previsoes_sempre_zero, zero_division=0),
    "F1-score": f1_score(y_desbalanceado, previsoes_sempre_zero, zero_division=0),
})

tabela_metricas_ingenuas = metricas_ingenuas.to_frame("modelo que sempre prevê 0")
tabela_metricas_ingenuas["modelo que sempre prevê 0"] = tabela_metricas_ingenuas[
    "modelo que sempre prevê 0"
].map(lambda valor: f"{valor:.2%}")
display(tabela_metricas_ingenuas)
print("\nMatriz de confusão:")
print(confusion_matrix(y_desbalanceado, previsoes_sempre_zero))


### Conclusão do experimento desbalanceado

O modelo ingênuo alcança **90% de acurácia**, mas tem **recall igual a zero** para a classe 1: ele não encontra nenhum atraso.

Antes de comemorar uma acurácia alta, sempre verifique:

1. a distribuição das classes;
2. a matriz de confusão;
3. as métricas da classe importante;
4. o custo dos falsos positivos e falsos negativos.


## 12. Desafio final — novas configurações

Crie duas novas versões dos modelos. Altere os parâmetros abaixo, execute a célula e compare os resultados com a primeira tabela.


In [ ]:
# SUA VEZ: altere estes valores.
K_ESCOLHIDO = 5
PROFUNDIDADE_ESCOLHIDA = 4

knn_desafio = KNeighborsClassifier(n_neighbors=K_ESCOLHIDO)
knn_desafio.fit(X_treino_escalado, y_treino)
previsoes_knn_desafio = knn_desafio.predict(X_teste_escalado)

arvore_desafio = DecisionTreeClassifier(
    max_depth=PROFUNDIDADE_ESCOLHIDA,
    random_state=RANDOM_STATE,
)
arvore_desafio.fit(X_treino, y_treino)
previsoes_arvore_desafio = arvore_desafio.predict(X_teste)

comparacao_desafio = pd.DataFrame([
    calcular_metricas(f"KNN (K = {K_ESCOLHIDO})", y_teste, previsoes_knn_desafio),
    calcular_metricas(
        f"Árvore (profundidade = {PROFUNDIDADE_ESCOLHIDA})",
        y_teste,
        previsoes_arvore_desafio,
    ),
]).set_index("Modelo")

comparacao_desafio_formatada = comparacao_desafio.map(lambda valor: f"{valor:.2%}")
display(comparacao_desafio_formatada)


## 13. Conclusão do grupo

Edite esta célula e responda:

1. Qual modelo teve maior acurácia?
2. Qual teve maior precisão?
3. Qual teve maior recall?
4. Qual teve maior F1-score?
5. Os modelos cometeram os mesmos tipos de erro?
6. Para o contexto estudado, qual erro custa mais: FP ou FN?
7. Qual modelo o grupo escolheria para o experimento e quais evidências sustentam a escolha?

**Conclusão:**  
Escreva aqui uma resposta argumentada. Evite escolher o modelo observando apenas uma métrica.


## 14. Exportando a tabela de comparação

Esta célula gera um CSV com as métricas dos dois modelos principais.


In [ ]:
nome_saida = "comparacao_metricas_modelos.csv"
comparacao_modelos.reset_index().to_csv(nome_saida, index=False)

print(f"Arquivo gerado: {nome_saida}")

# No Google Colab, retire o # das duas linhas abaixo para baixar o resultado:
# from google.colab import files
# files.download(nome_saida)


## Checklist de entrega

Seu notebook deve conter:

- [x] modelos treinados e previsões;
- [x] distribuição das classes;
- [x] acurácia;
- [x] matrizes de confusão;
- [x] precisão, recall e F1-score;
- [x] classification reports;
- [x] comparação entre KNN e Árvore de Decisão;
- [ ] parâmetros do desafio alterados pelo grupo;
- [ ] interpretação escrita e escolha argumentada do modelo.
